# Selecta Premiums Analysis

This notebook processes and analyses premium income data for the quarterly valuation process.

**Steps:**
1. Load premiums data
2. Clean and validate data
3. Summarise premiums by product and period
4. Calculate earned premiums
5. Export results for use in the quarterly valuation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Load Premiums Data

In [ ]:
# Load premiums data
# premiums_df = pd.read_csv('data/premiums.csv')
# Placeholder: create sample premiums data
premiums_df = pd.DataFrame({
    'policy_id': range(1000, 1200),
    'product': np.random.choice(['Term Life', 'Whole Life', 'Income Protection', 'Critical Illness'], 200),
    'start_date': pd.date_range('2020-01-01', periods=200, freq='W'),
    'expiry_date': pd.date_range('2025-01-01', periods=200, freq='W'),
    'annual_premium': np.random.uniform(500, 5000, 200).round(2),
    'payment_frequency': np.random.choice(['Monthly', 'Quarterly', 'Annual'], 200),
    'status': np.random.choice(['Active', 'Lapsed', 'Paid-Up'], 200, p=[0.8, 0.15, 0.05]),
})

print(f'Policies loaded: {len(premiums_df)} rows')
premiums_df.head()

## 2. Clean and Validate Data

In [ ]:
# Check for nulls
print('Null counts:')
print(premiums_df.isnull().sum())

# Check premium amounts are positive
assert (premiums_df['annual_premium'] > 0).all(), 'Non-positive premiums found'

# Check expiry is after start
assert (premiums_df['expiry_date'] > premiums_df['start_date']).all(), 'Expiry before start date found'

print('\nData validation passed.')

## 3. Summarise Premiums by Product

In [ ]:
summary = premiums_df.groupby(['product', 'status']).agg(
    policy_count=('policy_id', 'count'),
    total_annual_premium=('annual_premium', 'sum'),
).reset_index()

print(summary.to_string(index=False))

## 4. Calculate Earned Premiums

In [ ]:
# Filter to active policies only
active_df = premiums_df[premiums_df['status'] == 'Active'].copy()

# Quarterly earned premiums = annual premium / 4
active_df['quarterly_premium'] = active_df['annual_premium'] / 4

total_annual = active_df['annual_premium'].sum()
total_quarterly = active_df['quarterly_premium'].sum()
active_count = len(active_df)

print(f'Active Policies:          {active_count}')
print(f'Total Annual Premium:     £{total_annual:,.2f}')
print(f'Total Quarterly Premium:  £{total_quarterly:,.2f}')

## 5. Visualise Premiums

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

product_summary = premiums_df[premiums_df['status'] == 'Active'].groupby('product')['annual_premium'].sum()

# Premium income by product (bar)
product_summary.plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Annual Premium by Product (Active Policies)')
axes[0].set_xlabel('Product')
axes[0].set_ylabel('Premium (£)')
axes[0].tick_params(axis='x', rotation=15)

# Lapse rate by product (pie)
status_counts = premiums_df['status'].value_counts()
status_counts.plot.pie(ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Policy Status Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 6. Export Results

In [ ]:
# os.makedirs('output', exist_ok=True)
# summary.to_csv('output/premiums_summary.csv', index=False)
# active_df.to_csv('output/active_premiums.csv', index=False)
print('Premiums analysis complete. Results ready for quarterly valuation.')